# V2 standardized comparison: 2026-06-03 vs 2026-08-27

This notebook reduces known comparison variables before calculating improvement: identical 200 Hz low-pass bandwidth, 512 Hz sample rate, identical sample count within each upper/lower comparison, timestamp-integrity audit, and process-envelope alignment score. Only raw/unfiltered recordings are used for cross-date improvement. The vendor requirement is strictly |g| < 0.3 g at 265 wafers; available recordings are 250-wafer engineering evidence, not formal acceptance evidence.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str((Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()) / 'scripts'))
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from audit_data_comparability import standardize, envelope, align, TARGET_FS, COMMON_CUTOFF_HZ
from generate_vibration_report import load_recdata, calculate_metrics
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
DATASETS = {
    '2026-06-03 WPH249 upper/raw': Path(r'C:\ITRI\歐陽\SAA\WR503\震動量測\20260603\WPH249\上手\RecData--20260603142758.csv'),
    '2026-06-03 WPH250 upper/raw': Path(r'C:\ITRI\歐陽\SAA\WR503\震動量測\20260603\WPH250\上手\RecData--20260603153648.csv'),
    '2026-08-27 upper J5/raw': Path(r'C:\ITRI\歐陽\SAA\WR503\震動量測\20260827_vib test\濾波前\上手J5\RecData--20260827115016.csv'),
    '2026-06-03 WPH249 lower/raw': Path(r'C:\ITRI\歐陽\SAA\WR503\震動量測\20260603\WPH249\下手\RecData--20260603142937.csv'),
    '2026-06-03 WPH250 lower/raw': Path(r'C:\ITRI\歐陽\SAA\WR503\震動量測\20260603\WPH250\下手\RecData--20260603153829.csv'),
    '2026-08-27 lower J4/raw': Path(r'C:\ITRI\歐陽\SAA\WR503\震動量測\20260827_vib test\濾波前\下手J4\RecData--20260827115019.csv'),
}
GROUPS = {
    'Upper gripper': ['2026-06-03 WPH249 upper/raw', '2026-06-03 WPH250 upper/raw', '2026-08-27 upper J5/raw'],
    'Lower gripper': ['2026-06-03 WPH249 lower/raw', '2026-06-03 WPH250 lower/raw', '2026-08-27 lower J4/raw'],
}
THRESHOLD_G = 0.3
DISPLAY_MAX_POINTS = 60_000


In [ ]:
raw, metadata, audit_rows = {}, {}, []
for label, path in DATASETS.items():
    frame, meta = load_recdata(path)
    raw[label], metadata[label] = frame, meta
    dt = frame.time_s.diff().dropna().median()
    expected = round((frame.time_s.iloc[-1] - frame.time_s.iloc[0]) / dt) + 1
    audit_rows.append({
        'dataset': label, 'start': meta.get('Date/Time', 'missing'),
        'inferred_Hz': 1 / dt, 'samples': len(frame),
        'duration_s': frame.time_s.iloc[-1] - frame.time_s.iloc[0],
        'timeline_missing_samples': max(0, expected - len(frame)),
        'duplicate_timestamps': frame.time_s.duplicated().sum(),
        'large_time_gaps': (frame.time_s.diff() > 1.5 * dt).sum(),
    })
audit = pd.DataFrame(audit_rows).set_index('dataset')
display(audit.round(6))

standardized = {label: standardize(frame) for label, frame in raw.items()}
windows, window_rows = {}, []
for group, labels in GROUPS.items():
    common_samples = min(len(standardized[label]) for label in labels)
    for label in labels:
        windows[label] = standardized[label].iloc[:common_samples].copy()
    window_rows.append({'group': group, 'common_samples': common_samples, 'common_duration_s': (common_samples - 1) / TARGET_FS})
print(f'Common processing: 4th-order zero-phase low-pass {COMMON_CUTOFF_HZ:.0f} Hz; sample rate {TARGET_FS:.0f} Hz')
display(pd.DataFrame(window_rows).set_index('group'))


In [ ]:
alignment_rows = []
for group, labels in GROUPS.items():
    august = labels[-1]
    for benchmark in labels[:-1]:
        lag_s, correlation, overlap_s = align(envelope(standardized[august]), envelope(standardized[benchmark]))
        alignment_rows.append({
            'group': group, 'benchmark': benchmark, 'estimated_lag_s': lag_s, 'envelope_correlation': correlation,
            'overlap_s': overlap_s, 'phase_alignment_gate': 'PASS' if correlation >= 0.5 else 'FAIL',
        })
alignment_audit = pd.DataFrame(alignment_rows).set_index(['group','benchmark'])
display(alignment_audit.round(4))
print('FAIL means improvement remains conditional: timestamps alone cannot identify the same robot-motion/wafer-holding phase.')


In [ ]:
global_abs = max(frame[['X','Y','Z']].abs().to_numpy().max() for frame in windows.values())
y_limit = np.ceil(global_abs * 1.03 / 0.1) * 0.1
y_ticks = np.arange(-y_limit, y_limit + 0.1, 0.2)
for group, labels in GROUPS.items():
    fig, axs = plt.subplots(3, 1, figsize=(16, 10), sharex=True, sharey=True)
    for axis, ax in zip('XYZ', axs):
        for label in labels:
            frame = windows[label]
            step = max(1, int(np.ceil(len(frame) / DISPLAY_MAX_POINTS)))
            ax.plot(frame.time_s.iloc[::step], frame[axis].iloc[::step], lw=0.5, alpha=0.75, label=label)
        ax.axhline(THRESHOLD_G, color='red', ls='--', lw=1.1)
        ax.axhline(-THRESHOLD_G, color='red', ls='--', lw=1.1)
        ax.set_ylim(-y_limit, y_limit); ax.set_yticks(y_ticks); ax.set_ylabel(f'{axis} (g)')
    axs[0].legend(fontsize=8); axs[-1].set_xlabel('Standardized time (s)')
    fig.suptitle(f'{group}: standardized raw comparison (200 Hz bandwidth, 512 Hz, equal duration)')
    fig.tight_layout(); plt.show()


In [ ]:
metrics = calculate_metrics(windows)
display(metrics[['RMS_g','P99_abs_g','P99.9_abs_g','Max_abs_g','Within_pct','Exceed_pct','Events_per_min','Max_event_ms']].round(5))

LOWER_BETTER = ['RMS_g','P99_abs_g','P99.9_abs_g','Max_abs_g','Exceed_pct','Events_per_min','Max_event_ms']
rows = []
for group, labels in GROUPS.items():
    baseline_labels, august_label = labels[:-1], labels[-1]
    baseline = pd.concat([metrics.loc[label] for label in baseline_labels]).groupby(level=0).mean()
    august = metrics.loc[august_label]
    for axis in 'XYZ':
        row = {'group': group, 'axis': axis}
        for metric in LOWER_BETTER:
            old, new = baseline.loc[axis, metric], august.loc[axis, metric]
            row[f'{metric}_benchmark'] = old; row[f'{metric}_august'] = new
            row[f'{metric}_improvement_pct'] = 100 * (old - new) / old if old else np.nan
        rows.append(row)
improvement = pd.DataFrame(rows).set_index(['group','axis'])
for metric in LOWER_BETTER:
    print(f'\n{metric}')
    display(improvement[[f'{metric}_benchmark',f'{metric}_august',f'{metric}_improvement_pct']].round(4))

for metric in ['RMS_g','P99_abs_g','Exceed_pct','Events_per_min']:
    chart = improvement[f'{metric}_improvement_pct'].unstack('axis')
    ax = chart.plot.bar(figsize=(12,4.5)); ax.axhline(0,color='black',lw=.8)
    ax.set_ylabel('Improvement (%)'); ax.set_title(f'{metric}: positive = improvement, negative = regression')
    ax.tick_params(axis='x',rotation=0); plt.tight_layout(); plt.show()


## Interpretation boundary

The standardized metrics remove sampling-rate, bandwidth, and record-length differences. They do not prove that the same motion phase or wafer-holding interval was captured. When the phase-alignment gate fails, improvement percentages are conditional engineering indicators only. Formal validation requires synchronized GPST/motion-step timestamps and a repeated 265-wafer test.